In [2]:
"""
Cross-Modal LM-JEPA: Data Points ↔ Equation Structure
=======================================================
View A (Online):  Data points (x1,x2,x3,y) → DeepSets + CrossAttn → K vectors
View B (Target):  Token types sequence       → Transformer + CrossAttn → K vectors (EMA)

After training, the point encoder maps raw data → latent vectors that
capture symbolic equation structure, because the JEPA loss forced it to
match the equation encoder's representations.

Training strategy for 44k equations:
  Phase 1 (warmup, ~10 epochs):   High LR, high EMA momentum (0.99)
    - Both encoders learn basic representations fast
  Phase 2 (main, ~30 epochs):     Normal LR, EMA ramps 0.996→1.0
    - Point encoder refines to match equation encoder
  Phase 3 (finetune, ~10 epochs): Low LR, frozen EMA (tau=1.0)
    - Polish representations without moving target

Total: 50 epochs. Point encoder learns equation structure from data alone.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ==========================================
# 1. TOKEN VOCABULARY
# ==========================================

TOKEN_VOCAB = {
    "[PAD]": 0,
    "OP_ADD": 1, "OP_SUB": 2, "OP_MUL": 3, "OP_DIV": 4, "OP_POW": 5,
    "FUNC_SIN": 6, "FUNC_COS": 7, "FUNC_TAN": 8, "FUNC_EXP": 9,
    "FUNC_SQRT": 10, "FUNC_LOG": 11, "FUNC_NEG": 12, "FUNC_TANH": 13,
    "FUNC_ARCSIN": 14, "FUNC_ARCCOS": 15, "FUNC_ARCTAN": 16,
    "VAR": 17, "NUM": 18,
    "CONST_PI": 19, "CONST_E": 20,
    "[UNK]": 21,
}
VOCAB_SIZE = len(TOKEN_VOCAB)
MAX_SEQ_LEN = 64  # max tokens per equation (pad/truncate to this)


def tokenize_type_sequence(type_str):
    """Convert 'OP_ADD OP_SUB VAR ...' string → list of token IDs."""
    if not isinstance(type_str, str) or type_str.strip() == "":
        return []
    tokens = type_str.strip().split()
    return [TOKEN_VOCAB.get(t, TOKEN_VOCAB["[UNK]"]) for t in tokens]


# ==========================================
# 2. VALID INDICES + DATASET
# ==========================================

def get_valid_indices(npz_path, csv_path, total_expected=50000):
    """
    Return indices that are valid in BOTH npz (has data points)
    AND csv (has token_types).
    """
    data = np.load(npz_path)
    df = pd.read_csv(csv_path)

    valid = []
    print("Scanning for valid equations (points + tokens)...")

    for i in range(min(total_expected, len(df))):
        try:
            # check npz has data
            if f"X_{i}" not in data or data[f"X_{i}"].shape[0] == 0:
                continue
            # check csv has token_types
            tt = df.iloc[i].get("token_types", "")
            if not isinstance(tt, str) or tt.strip() == "":
                continue
            # check tokenization produces something
            ids = tokenize_type_sequence(tt)
            if len(ids) < 2:
                continue
            valid.append(i)
        except:
            continue

    print(f"Found {len(valid)} valid equations out of {total_expected}.")
    return valid


class CrossModalDataset(Dataset):
    """
    Preloads both:
      - Data points (B, 500, 4) from npz
      - Token type sequences from csv
    Returns two views for JEPA: point view + equation view.
    """

    def __init__(self, npz_path, csv_path, valid_indices):
        print("Loading data into RAM...")
        full_data = np.load(npz_path)
        df = pd.read_csv(csv_path)

        # Pre-allocate points
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.lengths = np.zeros(len(valid_indices), dtype=np.int32)

        # Pre-allocate token sequences
        self.token_ids = np.zeros((len(valid_indices), MAX_SEQ_LEN), dtype=np.int64)
        self.token_lens = np.zeros(len(valid_indices), dtype=np.int32)

        kept = 0
        for real_idx in tqdm(valid_indices, desc="Pre-loading"):
            try:
                # ── Points ──
                x_pts = np.array(full_data[f"X_{real_idx}"], dtype=np.float32)
                y_pts = np.array(full_data[f"y_{real_idx}"], dtype=np.float32).ravel()

                if x_pts.ndim == 1: x_pts = x_pts.reshape(-1, 1)
                n_p = min(x_pts.shape[0], len(y_pts), 500)
                n_v = min(x_pts.shape[1], 3)
                if n_p < 4: continue

                x, y = x_pts[:n_p, :n_v].copy(), y_pts[:n_p].copy()
                ok = np.isfinite(x).all(1) & np.isfinite(y)
                x, y = x[ok], y[ok]
                if len(y) < 4: continue

                for c in range(n_v):
                    s = x[:, c].std() + 1e-8
                    x[:, c] = (x[:, c] - x[:, c].mean()) / s
                ys = y.std() + 1e-8
                if ys < 1e-12: continue
                y = (y - y.mean()) / ys
                x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)
                n_p = len(y)

                # ── Tokens ──
                tt = df.iloc[real_idx]["expression_prefix_masked"]
                ids = tokenize_type_sequence(tt)
                if len(ids) < 2: continue
                seq_len = min(len(ids), MAX_SEQ_LEN)

                # Store
                self.points[kept, :n_p, :n_v] = x
                self.points[kept, :n_p, 3] = y
                self.lengths[kept] = n_p

                self.token_ids[kept, :seq_len] = ids[:seq_len]
                self.token_lens[kept] = seq_len
                kept += 1

            except:
                continue

        self.points = torch.from_numpy(self.points[:kept])
        self.lengths = torch.from_numpy(self.lengths[:kept])
        self.token_ids = torch.from_numpy(self.token_ids[:kept])
        self.token_lens = torch.from_numpy(self.token_lens[:kept])

        print(f"  Loaded {kept} equations")
        print(f"  Points shape: {self.points.shape}")
        print(f"  Token seqs shape: {self.token_ids.shape}")
        print(f"  Avg tokens/eq: {self.token_lens.float().mean():.1f}")

    def __len__(self):
        return len(self.points)

    def __getitem__(self, idx):
        # ── Point view (context) ──
        n = int(self.lengths[idx])
        pts = self.points[idx, :n]
        # use all points (no split — the other view is the equation)
        pts_pad = torch.zeros(500, 4)
        pts_mask = torch.zeros(500, dtype=torch.bool)
        pts_pad[:n] = pts
        pts_mask[:n] = True

        # ── Equation view (target) ──
        seq_len = int(self.token_lens[idx])
        tok_ids = self.token_ids[idx]  # already padded to MAX_SEQ_LEN
        tok_mask = torch.zeros(MAX_SEQ_LEN, dtype=torch.bool)
        tok_mask[:seq_len] = True

        return pts_pad, pts_mask, tok_ids, tok_mask


# ==========================================
# 3. POINT ENCODER: DeepSets + CrossAttn (online)
# ==========================================

class PointEncoder(nn.Module):
    """
    Same as your working DeepSets + CrossAttn baseline.
    φ per-point MLP → K queries cross-attend → K vectors.
    """

    def __init__(self, input_dim=4, hidden_dim=256, latent_dim=128, K=8, n_heads=4):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, latent_dim), nn.LayerNorm(latent_dim))

        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(latent_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.ffn = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2), nn.GELU(),
            nn.Linear(latent_dim * 2, latent_dim))

    def forward(self, x, mask=None):
        h = self.phi(x)
        q = self.queries.expand(x.size(0), -1, -1)
        key_pad = ~mask if mask is not None else None
        attn_out = self.cross_attn(self.norm1(q), h, h, key_padding_mask=key_pad)[0]
        q = q + attn_out
        q = q + self.ffn(self.norm2(q))
        return q  # (B, K, D)


# ==========================================
# 4. EQUATION ENCODER: Transformer + CrossAttn (EMA target)
# ==========================================

class EquationEncoder(nn.Module):
    """
    Token type sequence → Transformer → K queries cross-attend → K vectors.

    token_types → embed(vocab→128) → pos encoding → 2-layer Transformer
               → K learned queries cross-attend to token features → K×128
    """

    def __init__(self, vocab_size=VOCAB_SIZE, max_len=MAX_SEQ_LEN,
                 latent_dim=128, n_layers=2, n_heads=4, K=8, dropout=0.1):
        super().__init__()

        # Token embedding + positional encoding
        self.token_embed = nn.Embedding(vocab_size, latent_dim, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len, latent_dim)

        # Transformer self-attention on token sequence
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=latent_dim, nhead=n_heads,
                dim_feedforward=latent_dim * 2, dropout=dropout,
                activation="gelu", batch_first=True, norm_first=True),
            num_layers=n_layers)

        # K queries cross-attend to transformer output
        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(latent_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.ffn = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2), nn.GELU(),
            nn.Linear(latent_dim * 2, latent_dim))

    def forward(self, token_ids, mask=None):
        """
        token_ids: (B, max_len) long tensor
        mask:      (B, max_len) bool, True = valid token
        """
        B, L = token_ids.shape

        # Embed tokens + positions
        positions = torch.arange(L, device=token_ids.device).unsqueeze(0)
        h = self.token_embed(token_ids) + self.pos_embed(positions)  # (B, L, D)

        # Transformer (self-attention among tokens)
        key_pad = ~mask if mask is not None else None
        h = self.transformer(h, src_key_padding_mask=key_pad)        # (B, L, D)

        # K queries cross-attend to token features
        q = self.queries.expand(B, -1, -1)
        attn_out = self.cross_attn(self.norm1(q), h, h, key_padding_mask=key_pad)[0]
        q = q + attn_out
        q = q + self.ffn(self.norm2(q))

        return q  # (B, K, D)


# ==========================================
# 5. PREDICTOR
# ==========================================

class Predictor(nn.Module):
    """Maps K point vectors → K predicted equation vectors."""

    def __init__(self, dim=128, pred_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, pred_dim), nn.LayerNorm(pred_dim), nn.GELU(),
            nn.Linear(pred_dim, dim))

    def forward(self, z):
        return self.net(z)


# ==========================================
# 6. TRAINING
# ==========================================

def train_cross_modal_jepa(npz_file, csv_file, epochs=50, K=8, batch_size=512, lr=2e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── Data ──
    indices = get_valid_indices(npz_file, csv_file)
    dataset = CrossModalDataset(npz_file, csv_file, indices)
    n_train = int(0.9 * len(dataset)); n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    # ── Models ──
    # Online: point encoder (trained with gradients)
    point_encoder = PointEncoder(K=K).to(device)

    # Target: equation encoder (EMA, no gradients)
    eq_encoder = EquationEncoder(K=K).to(device)
    eq_encoder_ema = EquationEncoder(K=K).to(device)
    eq_encoder_ema.load_state_dict(eq_encoder.state_dict())
    for p in eq_encoder_ema.parameters():
        p.requires_grad = False

    # Predictor: maps point K vectors → predicted equation K vectors
    predictor = Predictor(dim=128, pred_dim=64).to(device)

    # ── Optimizer ──
    # Train point encoder + equation encoder + predictor together
    # EMA target is updated separately
    optimizer = torch.optim.AdamW([
        {"params": point_encoder.parameters(), "lr": lr},
        {"params": eq_encoder.parameters(), "lr": lr},
        {"params": predictor.parameters(), "lr": lr},
    ], weight_decay=0.05)

    # ── 3-phase schedule ──
    # Phase 1 (warmup):   epochs 1-10,  ramp up LR
    # Phase 2 (main):     epochs 11-40, cosine decay
    # Phase 3 (finetune): epochs 41-50, low LR, frozen EMA
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs,
        pct_start=0.2,    # 20% warmup (10 epochs)
        div_factor=10,    # start LR = max_lr / 10
        final_div_factor=100,  # end LR = max_lr / 1000
    )

    total_steps = epochs * len(train_loader)
    global_step = 0
    best_val = float("inf")

    pt_params = sum(p.numel() for p in point_encoder.parameters())
    eq_params = sum(p.numel() for p in eq_encoder.parameters())
    pr_params = sum(p.numel() for p in predictor.parameters())
    print(f"\nCross-Modal JEPA | K={K} | {device}")
    print(f"  Point encoder:  {pt_params:,} params")
    print(f"  Eq encoder:     {eq_params:,} params")
    print(f"  Predictor:      {pr_params:,} params")
    print(f"  Train: {n_train} | Val: {n_val} | Batch: {batch_size}")
    print(f"\n  Strategy: 3-phase (warmup→main→finetune)\n")

    for epoch in range(1, epochs + 1):

        # ── Determine training phase ──
        if epoch <= 10:
            phase = "warmup"
            ema_base = 0.99    # faster EMA during warmup
        elif epoch <= 40:
            phase = "main"
            ema_base = 0.996   # standard EMA
        else:
            phase = "finetune"
            ema_base = 1.0     # freeze EMA target

        # ── Train ──
        point_encoder.train()
        eq_encoder.train()
        predictor.train()
        total_loss, total_cos, total_std, n_batches = 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Ep {epoch}/{epochs} [{phase}]")
        for pts, pts_mask, tok_ids, tok_mask in pbar:
            pts      = pts.to(device, non_blocking=True)
            pts_mask = pts_mask.to(device, non_blocking=True)
            tok_ids  = tok_ids.to(device, non_blocking=True)
            tok_mask = tok_mask.to(device, non_blocking=True)

            # Online: point encoder → K vectors
            z_points = point_encoder(pts, pts_mask)           # (B, K, D)

            # Online: equation encoder (also trained, feeds into EMA)
            z_eq = eq_encoder(tok_ids, tok_mask)              # (B, K, D)

            # EMA target: equation encoder (no grad)
            with torch.no_grad():
                z_eq_target = eq_encoder_ema(tok_ids, tok_mask)  # (B, K, D)

            # Predictor: point vectors → predicted equation vectors
            z_predicted = predictor(z_points)                 # (B, K, D)

            # ── Loss ──
            pred_n = F.normalize(z_predicted, dim=-1, eps=1e-6)
            tgt_n  = F.normalize(z_eq_target, dim=-1, eps=1e-6)

            # Also align online eq encoder with its EMA (self-consistency)
            eq_n = F.normalize(z_eq, dim=-1, eps=1e-6)

            # Main loss: predicted point vectors ↔ EMA equation vectors
            loss_cross = F.smooth_l1_loss(pred_n, tgt_n.detach())

            # Auxiliary: online eq encoder ↔ EMA eq encoder (keeps eq encoder stable)
            loss_self = F.smooth_l1_loss(eq_n, tgt_n.detach()) * 0.5

            # Variance regularization (prevent collapse)
            loss_var = (F.relu(1.0 - pred_n.std(0).clamp(min=1e-6)).mean() +
                        F.relu(1.0 - eq_n.std(0).clamp(min=1e-6)).mean())

            loss = loss_cross + loss_self + loss_var

            if torch.isnan(loss):
                loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(point_encoder.parameters()) +
                list(eq_encoder.parameters()) +
                list(predictor.parameters()), 1.0)
            optimizer.step()
            scheduler.step()

            # ── EMA update equation encoder ──
            if ema_base < 1.0:
                # cosine schedule within current phase
                tau = 1.0 - (1.0 - ema_base) * (
                    math.cos(math.pi * global_step / total_steps) + 1) / 2
            else:
                tau = 1.0  # frozen in finetune phase

            with torch.no_grad():
                for p_t, p_o in zip(eq_encoder_ema.parameters(), eq_encoder.parameters()):
                    p_t.data.lerp_(p_o.data, 1.0 - tau)

            # Metrics
            with torch.no_grad():
                cos = F.cosine_similarity(
                    pred_n.flatten(0, 1), tgt_n.flatten(0, 1), dim=-1).mean()
                std = pred_n.std(0).mean()

            total_loss += loss.item()
            total_cos  += cos.item()
            total_std  += std.item()
            n_batches  += 1
            global_step += 1

            pbar.set_postfix({"loss": f"{loss.item():.4f}", "cos": f"{cos.item():.3f}"})

        # ── Validate ──
        point_encoder.eval()
        eq_encoder_ema.eval()
        predictor.eval()
        val_loss, val_cos, val_n = 0, 0, 0

        with torch.no_grad():
            for pts, pts_mask, tok_ids, tok_mask in val_loader:
                pts      = pts.to(device)
                pts_mask = pts_mask.to(device)
                tok_ids  = tok_ids.to(device)
                tok_mask = tok_mask.to(device)

                z_p = point_encoder(pts, pts_mask)
                z_t = eq_encoder_ema(tok_ids, tok_mask)
                z_pred = predictor(z_p)

                pn = F.normalize(z_pred, dim=-1, eps=1e-6)
                tn = F.normalize(z_t, dim=-1, eps=1e-6)

                vl = F.smooth_l1_loss(pn, tn) + F.relu(1.0 - pn.std(0).clamp(1e-6)).mean()
                vc = F.cosine_similarity(pn.flatten(0, 1), tn.flatten(0, 1), dim=-1).mean()

                val_loss += vl.item()
                val_cos  += vc.item()
                val_n    += 1

        # ── Log ──
        tl = total_loss / n_batches
        tc = total_cos / n_batches
        ts = total_std / n_batches
        vl = val_loss / val_n
        vc = val_cos / val_n

        if vl < best_val:
            best_val = vl

        warn = " ⚠ COLLAPSE" if ts < 0.01 else ""
        print(f"  loss={tl:.4f} cos={tc:.3f} std={ts:.3f} | "
              f"val={vl:.4f} val_cos={vc:.3f} | τ={tau:.4f}{warn}")

    # ── Save ──
    torch.save({
        "point_encoder": point_encoder.state_dict(),
        "eq_encoder": eq_encoder.state_dict(),
        "eq_encoder_ema": eq_encoder_ema.state_dict(),
        "predictor": predictor.state_dict(),
        "val_loss": best_val, "K": K,
    }, "jepa_cross_modal.pt")

    print(f"\nDone! Best val={best_val:.4f}")
    print("Saved: jepa_cross_modal.pt")
    print("\nTo use the trained point encoder for inference:")
    print("  z = point_encoder(new_data_points, mask)  # → K×128 equation-aware vectors")


# ── Inference helper ──────────────────────────────────────

@torch.no_grad()
def encode_points(checkpoint_path, points, mask=None, K=8, device="cuda"):
    """
    Load trained point encoder and encode new data points.
    points: (B, N, 4) tensor
    mask:   (B, N) bool tensor
    Returns: (B, K, 128) equation-aware latent vectors
    """
    ckpt = torch.load(checkpoint_path, map_location=device)
    encoder = PointEncoder(K=K).to(device)
    encoder.load_state_dict(ckpt["point_encoder"])
    encoder.eval()
    return encoder(points.to(device), mask.to(device) if mask is not None else None)


# --- RUN ---
train_cross_modal_jepa(
    npz_file="/content/drive/MyDrive/Colab Notebooks/18march 50k data/Copy of equations_50k_data.npz",
    csv_file="/content/drive/MyDrive/Colab Notebooks/18march 50k data/Copy of equations_50k_final_input_18march.csv",
    epochs=50,
    K=8,
    batch_size=512,
    lr=2e-3,
)

Scanning for valid equations (points + tokens)...
Found 44820 valid equations out of 50000.
Loading data into RAM...


Pre-loading: 100%|██████████| 44820/44820 [03:01<00:00, 247.18it/s]
/tmp/ipykernel_923/954110290.py:246: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  Loaded 44820 equations
  Points shape: torch.Size([44820, 500, 4])
  Token seqs shape: torch.Size([44820, 64])
  Avg tokens/eq: 14.7

Cross-Modal JEPA | K=8 | cuda
  Point encoder:  234,752 params
  Eq encoder:     409,472 params
  Predictor:      16,704 params
  Train: 40338 | Val: 4482 | Batch: 512

  Strategy: 3-phase (warmup→main→finetune)



Ep 1/50 [warmup]: 100%|██████████| 78/78 [00:16<00:00,  4.68it/s, loss=1.8327, cos=0.041]


  loss=1.8473 cos=0.024 std=0.082 | val=0.9195 val_cos=0.045 | τ=0.9900


...

  loss=1.8295 cos=0.194 std=0.088 | val=0.9180 val_cos=0.175 | τ=1.0000


Ep 49/50 [finetune]: 100%|██████████| 78/78 [00:18<00:00,  4.24it/s, loss=1.8295, cos=0.188]


  loss=1.8295 cos=0.193 std=0.088 | val=0.9180 val_cos=0.175 | τ=1.0000


Ep 50/50 [finetune]: 100%|██████████| 78/78 [00:18<00:00,  4.26it/s, loss=1.8292, cos=0.220]


  loss=1.8295 cos=0.193 std=0.088 | val=0.9180 val_cos=0.175 | τ=1.0000

Done! Best val=0.9180
Saved: jepa_cross_modal.pt

To use the trained point encoder for inference:
  z = point_encoder(new_data_points, mask)  # → K×128 equation-aware vectors


In [3]:
"""
Cross-Modal LM-JEPA with CLIP-style Alignment
================================================
JEPA flavor: predict equation embeddings from point embeddings
CLIP flavor: contrastive loss in a shared projection space

Architecture:
  Point Encoder  → K=8 vectors → pool → projection head → shared space (256d)
  Eq Encoder     → K=8 vectors → pool → projection head → shared space (256d)
                                                                ↕
                                          Contrastive + Predictive loss

  Point Encoder  → K=8 vectors → cross-attn predictor → predicted eq vectors
  EMA Eq Encoder → K=8 vectors ← ──── JEPA loss (predicted vs actual)

Three losses:
  1. Contrastive (CLIP):  this equation's points ↔ this equation's tokens (shared space)
  2. Predictive (JEPA):   predict eq K-vectors from point K-vectors (embedding space)
  3. Variance:            prevent collapse
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ==========================================
# 1. VOCAB + TOKENIZER
# ==========================================

BINARY_SET = {"ADD", "SUB", "MUL", "DIV", "POW"}
UNARY_SET  = {"sin", "cos", "tan", "exp", "sqrt", "log", "NEG", "tanh", "arcsin", "arccos", "arctan"}
SPECIAL    = {"pi", "e"}
VARIABLES  = {"x1", "x2", "x3"}

# Use ORIGINAL prefix tokens — not the lossy token_types
# Build vocab dynamically from data, but seed with known tokens
SEED_VOCAB = ["[PAD]", "[UNK]",
              "ADD", "SUB", "MUL", "DIV", "POW",
              "sin", "cos", "tan", "exp", "sqrt", "log", "NEG", "tanh",
              "arcsin", "arccos", "arctan",
              "x1", "x2", "x3", "pi", "e"]

MAX_SEQ_LEN = 40


def build_vocab(df, expr_col="expression_prefix", max_num_tokens=200):
    """Build vocab from actual expressions. Numbers get bucketed."""
    vocab = {t: i for i, t in enumerate(SEED_VOCAB)}

    # scan for any new tokens + bucket numbers
    for expr in df[expr_col].dropna():
        for tok in str(expr).strip().split():
            if tok in vocab:
                continue
            try:
                float(tok)
                if "NUM" not in vocab:
                    vocab["NUM"] = len(vocab)
            except ValueError:
                if len(vocab) < max_num_tokens:
                    vocab[tok] = len(vocab)

    if "NUM" not in vocab:
        vocab["NUM"] = len(vocab)

    print(f"Vocab size: {len(vocab)}")
    return vocab


def tokenize(expr_str, vocab):
    """Convert prefix expression → token IDs using full vocab."""
    if not isinstance(expr_str, str) or expr_str.strip() == "":
        return []
    ids = []
    for tok in expr_str.strip().split():
        if tok in vocab:
            ids.append(vocab[tok])
        else:
            try:
                float(tok)
                ids.append(vocab["NUM"])
            except ValueError:
                ids.append(vocab["[UNK]"])
    return ids


# ==========================================
# 2. DATASET
# ==========================================

def get_valid_indices(npz_path, csv_path, total_expected=50000):
    data = np.load(npz_path)
    df = pd.read_csv(csv_path)
    valid = []
    print("Scanning for valid equations...")
    for i in range(min(total_expected, len(df))):
        try:
            if f"X_{i}" not in data or data[f"X_{i}"].shape[0] == 0:
                continue
            expr = df.iloc[i].get("expression_prefix", "")
            if not isinstance(expr, str) or len(expr.strip().split()) < 2:
                continue
            valid.append(i)
        except:
            continue
    print(f"Found {len(valid)} valid equations.")
    return valid


class CrossModalDataset(Dataset):
    def __init__(self, npz_path, csv_path, valid_indices, vocab):
        print("Loading data into RAM...")
        full_data = np.load(npz_path)
        df = pd.read_csv(csv_path)

        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.lengths = np.zeros(len(valid_indices), dtype=np.int32)
        self.token_ids = np.zeros((len(valid_indices), MAX_SEQ_LEN), dtype=np.int64)
        self.token_lens = np.zeros(len(valid_indices), dtype=np.int32)

        kept = 0
        for real_idx in tqdm(valid_indices, desc="Pre-loading"):
            try:
                x_pts = np.array(full_data[f"X_{real_idx}"], dtype=np.float32)
                y_pts = np.array(full_data[f"y_{real_idx}"], dtype=np.float32).ravel()
            except: continue

            if x_pts.ndim == 1: x_pts = x_pts.reshape(-1, 1)
            n_p = min(x_pts.shape[0], len(y_pts), 500)
            n_v = min(x_pts.shape[1], 3)
            if n_p < 4: continue
            x, y = x_pts[:n_p, :n_v].copy(), y_pts[:n_p].copy()
            ok = np.isfinite(x).all(1) & np.isfinite(y)
            x, y = x[ok], y[ok]
            if len(y) < 4: continue
            for c in range(n_v):
                s = x[:, c].std() + 1e-8
                x[:, c] = (x[:, c] - x[:, c].mean()) / s
            ys = y.std() + 1e-8
            if ys < 1e-12: continue
            y = (y - y.mean()) / ys
            x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)
            n_p = len(y)

            expr = df.iloc[real_idx].get("expression_prefix", "")
            ids = tokenize(str(expr), vocab)
            if len(ids) < 2: continue
            seq_len = min(len(ids), MAX_SEQ_LEN)

            self.points[kept, :n_p, :n_v] = x
            self.points[kept, :n_p, 3] = y
            self.lengths[kept] = n_p
            self.token_ids[kept, :seq_len] = ids[:seq_len]
            self.token_lens[kept] = seq_len
            kept += 1

        self.points = torch.from_numpy(self.points[:kept])
        self.lengths = torch.from_numpy(self.lengths[:kept])
        self.token_ids = torch.from_numpy(self.token_ids[:kept])
        self.token_lens = torch.from_numpy(self.token_lens[:kept])
        print(f"  Loaded {kept} equations")

    def __len__(self): return len(self.points)

    def __getitem__(self, idx):
        n = int(self.lengths[idx])
        pts_pad = torch.zeros(500, 4)
        pts_mask = torch.zeros(500, dtype=torch.bool)
        pts_pad[:n] = self.points[idx, :n]
        pts_mask[:n] = True

        seq_len = int(self.token_lens[idx])
        tok_ids = self.token_ids[idx]
        tok_mask = torch.zeros(MAX_SEQ_LEN, dtype=torch.bool)
        tok_mask[:seq_len] = True

        return pts_pad, pts_mask, tok_ids, tok_mask


# ==========================================
# 3. POINT ENCODER (DeepSets + CrossAttn)
# ==========================================

class PointEncoder(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=128, latent_dim=128, K=8, n_heads=4):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, latent_dim), nn.LayerNorm(latent_dim))
        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(latent_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.ffn = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2), nn.GELU(),
            nn.Linear(latent_dim * 2, latent_dim))

    def forward(self, x, mask=None):
        h = self.phi(x)
        q = self.queries.expand(x.size(0), -1, -1)
        kp = ~mask if mask is not None else None
        q = q + self.cross_attn(self.norm1(q), h, h, key_padding_mask=kp)[0]
        q = q + self.ffn(self.norm2(q))
        return q  # (B, K, D)


# ==========================================
# 4. EQUATION ENCODER (Transformer + CrossAttn)
# ==========================================

class EquationEncoder(nn.Module):
    def __init__(self, vocab_size, max_len=MAX_SEQ_LEN,
                 latent_dim=128, n_layers=2, n_heads=4, K=8, dropout=0.1):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, latent_dim, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len, latent_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=latent_dim, nhead=n_heads,
                dim_feedforward=latent_dim * 2, dropout=dropout,
                activation="gelu", batch_first=True, norm_first=True),
            num_layers=n_layers)
        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(latent_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.ffn = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2), nn.GELU(),
            nn.Linear(latent_dim * 2, latent_dim))

    def forward(self, token_ids, mask=None):
        B, L = token_ids.shape
        pos = torch.arange(L, device=token_ids.device).unsqueeze(0)
        h = self.token_embed(token_ids) + self.pos_embed(pos)
        kp = ~mask if mask is not None else None
        h = self.transformer(h, src_key_padding_mask=kp)
        q = self.queries.expand(B, -1, -1)
        q = q + self.cross_attn(self.norm1(q), h, h, key_padding_mask=kp)[0]
        q = q + self.ffn(self.norm2(q))
        return q  # (B, K, D)


# ==========================================
# 5. PROJECTION HEADS (into shared CLIP space)
# ==========================================

class ProjectionHead(nn.Module):
    """Projects K vectors → single normalized vector in shared space."""
    def __init__(self, latent_dim=128, proj_dim=256, K=8):
        super().__init__()
        self.pool = nn.Sequential(
            nn.Linear(latent_dim * K, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Linear(proj_dim, proj_dim),
        )
    def forward(self, z):
        # z: (B, K, D) → flatten → project → normalize
        B = z.size(0)
        return F.normalize(self.pool(z.reshape(B, -1)), dim=-1)  # (B, proj_dim)


# ==========================================
# 6. CROSS-ATTENTION PREDICTOR (JEPA part)
# ==========================================

class CrossAttnPredictor(nn.Module):
    """
    Predicts equation K-vectors from point K-vectors.
    Uses cross-attention so each predicted eq vector
    can attend to ALL point vectors (not just 1-to-1).
    """
    def __init__(self, latent_dim=128, K=8, n_heads=4):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(latent_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.ffn = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2), nn.GELU(),
            nn.Linear(latent_dim * 2, latent_dim))
        # learnable target queries for prediction
        self.pred_queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)

    def forward(self, z_points):
        # z_points: (B, K, D) — point encoder output
        B = z_points.size(0)
        q = self.pred_queries.expand(B, -1, -1)
        q = q + self.cross_attn(self.norm1(q), z_points, z_points)[0]
        q = q + self.ffn(self.norm2(q))
        return q  # (B, K, D)


# ==========================================
# 7. CONTRASTIVE LOSS (CLIP-style)
# ==========================================

def contrastive_loss(z_pts, z_eq, temperature=0.07):
    """
    CLIP-style InfoNCE: match point embeddings to equation embeddings.
    z_pts, z_eq: (B, proj_dim) — L2-normalized projected vectors
    """
    # cosine similarity matrix
    logits = torch.mm(z_pts, z_eq.t()) / temperature  # (B, B)
    labels = torch.arange(len(z_pts), device=z_pts.device)
    # symmetric loss
    loss_p2e = F.cross_entropy(logits, labels)
    loss_e2p = F.cross_entropy(logits.t(), labels)
    return (loss_p2e + loss_e2p) / 2


# ==========================================
# 8. TRAINING
# ==========================================

def train(npz_file, csv_file, epochs=50, K=8, batch_size=512, lr=2e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── Data ──
    df = pd.read_csv(csv_file)
    vocab = build_vocab(df)
    indices = get_valid_indices(npz_file, csv_file)
    dataset = CrossModalDataset(npz_file, csv_file, indices, vocab)

    n_train = int(0.9 * len(dataset)); n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    # ── Models ──
    point_enc = PointEncoder(K=K).to(device)
    eq_enc = EquationEncoder(len(vocab), K=K).to(device)

    # EMA copy of equation encoder (for JEPA predictive loss)
    eq_enc_ema = EquationEncoder(len(vocab), K=K).to(device)
    eq_enc_ema.load_state_dict(eq_enc.state_dict())
    for p in eq_enc_ema.parameters(): p.requires_grad = False

    # Projection heads (for contrastive loss)
    proj_pts = ProjectionHead(128, 256, K).to(device)
    proj_eq = ProjectionHead(128, 256, K).to(device)

    # Cross-attention predictor (for JEPA predictive loss)
    predictor = CrossAttnPredictor(128, K).to(device)

    # ── Optimizer ──
    all_params = (list(point_enc.parameters()) + list(eq_enc.parameters()) +
                  list(proj_pts.parameters()) + list(proj_eq.parameters()) +
                  list(predictor.parameters()))
    optimizer = torch.optim.AdamW(all_params, lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs,
        pct_start=0.15, div_factor=10, final_div_factor=100)

    total_steps = epochs * len(train_loader)
    global_step = 0; best_val = float("inf")

    pt_p = sum(p.numel() for p in point_enc.parameters())
    eq_p = sum(p.numel() for p in eq_enc.parameters())
    print(f"\nJEPA-CLIP | K={K} | {device}")
    print(f"  Point enc: {pt_p:,} | Eq enc: {eq_p:,} | Vocab: {len(vocab)}")
    print(f"  Train: {n_train} | Val: {n_val} | Batch: {batch_size}\n")

    for epoch in range(1, epochs + 1):
        point_enc.train(); eq_enc.train()
        proj_pts.train(); proj_eq.train(); predictor.train()
        sl, sc_clip, sc_jepa, ss, nb = 0, 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Ep {epoch}/{epochs}")
        for pts, pts_mask, tok_ids, tok_mask in pbar:
            pts      = pts.to(device, non_blocking=True)
            pts_mask = pts_mask.to(device, non_blocking=True)
            tok_ids  = tok_ids.to(device, non_blocking=True)
            tok_mask = tok_mask.to(device, non_blocking=True)

            # ── Encode ──
            z_pts_k = point_enc(pts, pts_mask)              # (B, K, D)
            z_eq_k = eq_enc(tok_ids, tok_mask)              # (B, K, D)

            with torch.no_grad():
                z_eq_ema_k = eq_enc_ema(tok_ids, tok_mask)  # (B, K, D)

            # ── Loss 1: Contrastive (CLIP) in shared projection space ──
            z_pts_proj = proj_pts(z_pts_k)                  # (B, 256) normalized
            z_eq_proj = proj_eq(z_eq_k)                     # (B, 256) normalized
            loss_clip = contrastive_loss(z_pts_proj, z_eq_proj)

            # ── Loss 2: Predictive (JEPA) in K-vector space ──
            z_pred_k = predictor(z_pts_k)                   # (B, K, D)
            pred_n = F.normalize(z_pred_k, dim=-1, eps=1e-6)
            tgt_n = F.normalize(z_eq_ema_k, dim=-1, eps=1e-6)
            loss_jepa = F.smooth_l1_loss(pred_n, tgt_n.detach())

            # ── Loss 3: Variance (prevent collapse) ──
            loss_var = (F.relu(1.0 - z_pts_proj.std(0).clamp(min=1e-6)).mean() +
                        F.relu(1.0 - z_eq_proj.std(0).clamp(min=1e-6)).mean())

            # ── Total ──
            loss = loss_clip + 0.5 * loss_jepa + 0.5 * loss_var

            if torch.isnan(loss):
                loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(all_params, 1.0)
            optimizer.step(); scheduler.step()

            # EMA update
            tau = 1.0 - (1.0 - 0.996) * (math.cos(math.pi * global_step / total_steps) + 1) / 2
            with torch.no_grad():
                for p_t, p_o in zip(eq_enc_ema.parameters(), eq_enc.parameters()):
                    p_t.data.lerp_(p_o.data, 1.0 - tau)

            # Metrics
            with torch.no_grad():
                clip_cos = F.cosine_similarity(z_pts_proj, z_eq_proj, -1).mean()
                jepa_cos = F.cosine_similarity(pred_n.flatten(0,1), tgt_n.flatten(0,1), -1).mean()

            sl += loss.item(); sc_clip += clip_cos.item()
            sc_jepa += jepa_cos.item(); ss += z_pts_proj.std(0).mean().item()
            nb += 1; global_step += 1
            pbar.set_postfix({"loss": f"{loss.item():.3f}",
                              "clip": f"{clip_cos.item():.3f}",
                              "jepa": f"{jepa_cos.item():.3f}"})

        # ── Validate ──
        point_enc.eval(); eq_enc_ema.eval()
        proj_pts.eval(); proj_eq.eval(); predictor.eval()
        vl, vc_clip, vc_jepa, vn = 0, 0, 0, 0

        with torch.no_grad():
            for pts, pts_mask, tok_ids, tok_mask in val_loader:
                pts = pts.to(device); pts_mask = pts_mask.to(device)
                tok_ids = tok_ids.to(device); tok_mask = tok_mask.to(device)

                z_p = point_enc(pts, pts_mask)
                z_e = eq_enc_ema(tok_ids, tok_mask)

                zpp = proj_pts(z_p); zep = proj_eq(z_e)
                v_clip = contrastive_loss(zpp, zep)
                v_clip_cos = F.cosine_similarity(zpp, zep, -1).mean()

                z_pred = predictor(z_p)
                pn = F.normalize(z_pred, -1, eps=1e-6)
                tn = F.normalize(z_e, -1, eps=1e-6)
                v_jepa_cos = F.cosine_similarity(pn.flatten(0,1), tn.flatten(0,1), -1).mean()

                vl += v_clip.item(); vc_clip += v_clip_cos.item()
                vc_jepa += v_jepa_cos.item(); vn += 1

        tl = sl/nb; tc = sc_clip/nb; tj = sc_jepa/nb; ts = ss/nb
        vloss = vl/vn; vcc = vc_clip/vn; vcj = vc_jepa/vn
        if vloss < best_val: best_val = vloss

        warn = " ⚠COLLAPSE" if ts < 0.01 else ""
        print(f"  loss={tl:.3f} clip_cos={tc:.3f} jepa_cos={tj:.3f} std={ts:.3f} | "
              f"val_clip={vcc:.3f} val_jepa={vcj:.3f}{warn}")

    torch.save({
        "point_encoder": point_enc.state_dict(),
        "eq_encoder": eq_enc.state_dict(),
        "eq_encoder_ema": eq_enc_ema.state_dict(),
        "proj_pts": proj_pts.state_dict(),
        "proj_eq": proj_eq.state_dict(),
        "predictor": predictor.state_dict(),
        "vocab": vocab,
        "val_loss": best_val, "K": K,
    }, "jepa_clip.pt")
    print(f"\nDone! Best val={best_val:.4f}")


# --- RUN ---
train(
    npz_file="/content/drive/MyDrive/Colab Notebooks/18march 50k data/Copy of equations_50k_data.npz",
    csv_file="/content/drive/MyDrive/Colab Notebooks/18march 50k data/Copy of equations_50k_final_input_18march.csv",
    epochs=50,
    K=8,
    batch_size=512,
    lr=2e-3,
)

Vocab size: 29
Scanning for valid equations...
Found 44820 valid equations.
Loading data into RAM...


Pre-loading: 100%|██████████| 44820/44820 [03:00<00:00, 247.94it/s]
/tmp/ipykernel_923/984343676.py:223: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  Loaded 44820 equations

JEPA-CLIP | K=8 | cuda
  Point enc: 167,936 | Eq enc: 407,296 | Vocab: 29
  Train: 40338 | Val: 4482 | Batch: 512



Ep 1/50: 100%|██████████| 78/78 [00:12<00:00,  6.37it/s, loss=6.028, clip=0.492, jepa=0.674]


  loss=6.468 clip_cos=0.303 jepa_cos=0.599 std=0.037 | val_clip=0.360 val_jepa=0.094


...

  loss=2.621 clip_cos=0.643 jepa_cos=0.368 std=0.062 | val_clip=0.385 val_jepa=0.063


Ep 49/50: 100%|██████████| 78/78 [00:12<00:00,  6.02it/s, loss=2.659, clip=0.639, jepa=0.368]


  loss=2.618 clip_cos=0.643 jepa_cos=0.368 std=0.062 | val_clip=0.385 val_jepa=0.062


Ep 50/50: 100%|██████████| 78/78 [00:12<00:00,  6.00it/s, loss=2.534, clip=0.650, jepa=0.373]


  loss=2.610 clip_cos=0.644 jepa_cos=0.368 std=0.062 | val_clip=0.385 val_jepa=0.063

Done! Best val=4.1495
